# 006 · Encoding and Hashing with base64 and hashlib

> **Standard Library › Data Formats** · Python 3.14

Two jobs that often appear together: **encoding** binary data as text (Base64, so it can travel in JSON, URLs or headers) and **hashing** data into a fixed-size fingerprint (checksums, cache keys, signatures). `base64`, `hashlib` and `hmac` cover both — and `hashlib` also provides proper password hashing.

**In this notebook you will learn:**

- `base64.b64encode`/`b64decode` and the URL-safe variant — and the bytes-in/bytes-out gotcha
- hashing with `sha256`, `blake2b` and `md5`, as `digest()` or `hexdigest()`
- hashing large files incrementally, and with `hashlib.file_digest` (3.11+)
- message authentication with `hmac` and constant-time comparison
- password hashing with `scrypt` / `pbkdf2_hmac` — and why plain SHA-256 is wrong for passwords

## .NET mental model

| .NET | Python |
|---|---|
| `Convert.ToBase64String(bytes)` | `base64.b64encode(data)` → **bytes** |
| `Convert.FromBase64String(s)` | `base64.b64decode(s)` → bytes |
| `Base64Url.EncodeToString(bytes)` (.NET 9, `System.Buffers.Text`) | `base64.urlsafe_b64encode(data)` (keeps `=` padding) |
| `SHA256.HashData(bytes)` | `hashlib.sha256(data).digest()` |
| `Convert.ToHexStringLower(hash)` | `.hexdigest()` |
| `IncrementalHash` / `SHA256.HashData(stream)` | `h.update(chunk)` / `hashlib.file_digest(f, "sha256")` |
| `MD5.HashData(bytes)` | `hashlib.md5(data)` |
| — (no BCL BLAKE2) | `hashlib.blake2b(data)` |
| `HMACSHA256.HashData(key, data)` | `hmac.digest(key, data, "sha256")` |
| `CryptographicOperations.FixedTimeEquals(a, b)` | `hmac.compare_digest(a, b)` |
| `Rfc2898DeriveBytes.Pbkdf2(...)` | `hashlib.pbkdf2_hmac(...)` |
| — (no BCL scrypt) | `hashlib.scrypt(...)` |

## Base64: encoding and decoding

```csharp
// C#
string encoded = Convert.ToBase64String(Encoding.UTF8.GetBytes("héllo"));
string decoded = Encoding.UTF8.GetString(Convert.FromBase64String(encoded));
```

In Python, Base64 works on **bytes** — encode your text to bytes first *(see Standard Library › Text Processing › 004 · Unicode, Bytes and Encodings)*.

In [ ]:
import base64

raw = "héllo".encode("utf-8")
encoded = base64.b64encode(raw)
print(encoded)                          # b'...' — bytes, not str!

decoded = base64.b64decode(encoded)
print(decoded, decoded.decode("utf-8"))

⚠️ **Gotcha:** `b64encode` returns **bytes**, not `str` — unlike `Convert.ToBase64String`. Putting it straight into an f-string or JSON gives you `"b'aMOpbGxv'"`. Call `.decode("ascii")` to get text. Passing a `str` *in* is an error.

In [ ]:
print(f"token={encoded}")                    # oops: b'...' leaks into the text
print(f"token={encoded.decode('ascii')}")    # correct

try:
    base64.b64encode("héllo")                # str instead of bytes
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

`b64decode` accepts either `str` or `bytes`. By default it silently **discards** characters outside the Base64 alphabet; pass `validate=True` to be strict. Bad input raises `binascii.Error`, a subclass of `ValueError`.

In [ ]:
print(base64.b64decode("aGVs bG8="))                   # space silently ignored

for bad in ["aGVsbG8", "aGVs!bG8="]:
    try:
        base64.b64decode(bad, validate=True)
    except ValueError as e:
        print(f"{type(e).__name__}: {e}")

## URL-safe Base64

Standard Base64 uses `+` and `/`, which clash with URLs and file names. The URL-safe alphabet uses `-` and `_` instead (the `Base64Url` class in .NET 9, or `WebEncoders.Base64UrlEncode` in ASP.NET Core).

⚠️ **Gotcha:** Python's `urlsafe_b64encode` keeps the `=` padding, whereas `Base64Url` / JWTs omit it. Strip it when encoding and add it back before decoding.

In [ ]:
data = bytes([0xFB, 0xFF, 0xBF, 0x01])

print(base64.b64encode(data))            # contains + and /
print(base64.urlsafe_b64encode(data))    # - and _ instead, still padded

token = base64.urlsafe_b64encode(data).rstrip(b"=").decode("ascii")
print(token)                             # unpadded, JWT style

padded = token + "=" * (-len(token) % 4)   # restore padding to a multiple of 4
print(base64.urlsafe_b64decode(padded) == data)

**Tip:** the module also has `b32encode`, `b16encode` (hex), `a85encode`, `b85encode` and `z85encode` (3.13+). For plain hex, `bytes.hex()` and `bytes.fromhex()` are simpler.

## Hashing with `hashlib`

A hash object is created with a constructor (`hashlib.sha256(...)`), fed bytes, and asked for `digest()` (raw bytes) or `hexdigest()` (lowercase hex string).

```csharp
// C#
byte[] hash = SHA256.HashData(Encoding.UTF8.GetBytes("hello"));
string hex = Convert.ToHexStringLower(hash);   // .NET 9; earlier: ToHexString(...).ToLowerInvariant()
```

In [ ]:
import hashlib

h = hashlib.sha256(b"hello")
print(h.hexdigest())
print(len(h.digest()), h.digest_size, h.name)

# hashing text: encode first
print(hashlib.sha256("héllo".encode("utf-8")).hexdigest())

Other algorithms work the same way. `algorithms_guaranteed` lists those available on every platform; `hashlib.new(name)` picks one by name (like `HashAlgorithmName`).

In [ ]:
print(sorted(hashlib.algorithms_guaranteed))

message = b"The quick brown fox"
print("md5    ", hashlib.md5(message).hexdigest())
print("sha1   ", hashlib.sha1(message).hexdigest())
print("sha512 ", hashlib.sha512(message).hexdigest()[:32], "...")
print("blake2b", hashlib.blake2b(message, digest_size=16).hexdigest())
print("new()  ", hashlib.new("sha3_256", message).hexdigest()[:32], "...")

- **SHA-256** — the safe default for integrity checks and fingerprints.
- **BLAKE2b** — fast, modern, with configurable `digest_size`, built-in `key=` (keyed hashing) and `salt=`/`person=` parameters. No .NET BCL equivalent.
- **MD5 / SHA-1** — broken for security; fine only for non-security checksums. Python lets you say so explicitly with `usedforsecurity=False` (3.9+), which also keeps it working on FIPS-restricted systems.

In [ ]:
print(hashlib.md5(b"cache-key:product:42", usedforsecurity=False).hexdigest())

## Hashing large data incrementally

`update()` can be called repeatedly — hashing `a` then `b` equals hashing `a + b`. Use it to hash big files in chunks without loading them into memory (like `IncrementalHash.AppendData`).

In [ ]:
h1 = hashlib.sha256()
h1.update(b"hello ")
h1.update(b"world")
print(h1.hexdigest() == hashlib.sha256(b"hello world").hexdigest())

In [ ]:
import tempfile
from pathlib import Path

def sha256_of_file(path: Path, chunk_size: int = 64 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "big.bin"
    path.write_bytes(b"sensor-data " * 200_000)    # ~2.4 MB
    print(path.stat().st_size)
    print(sha256_of_file(path))

    with path.open("rb") as f:                     # the 3.11+ one-liner
        print(hashlib.file_digest(f, "sha256").hexdigest())

`hashlib.file_digest(fileobj, digest)` (Python 3.11+) does the chunked loop for you — the counterpart of `SHA256.HashData(Stream)`. The file must be opened in **binary** mode.

## Message authentication with `hmac`

A plain hash proves nothing about *who* produced the data — anyone can recompute it. An **HMAC** mixes in a secret key, so only key holders can produce a valid tag. Typical uses: webhook signatures, signed cookies, tamper-proof tokens.

```csharp
// C#
byte[] tag = HMACSHA256.HashData(key, payload);
bool ok = CryptographicOperations.FixedTimeEquals(tag, receivedTag);
```

In [ ]:
import hmac

key = b"demo-key-not-a-real-secret"
payload = b'{"order_id": 1001, "total": "19.99"}'

tag = hmac.new(key, payload, hashlib.sha256).hexdigest()   # digestmod is required
print(tag)

one_shot = hmac.digest(key, payload, "sha256")             # fast one-shot, returns bytes
print(one_shot.hex() == tag)

To **verify**, recompute the tag and compare with `hmac.compare_digest`, which takes the same time no matter where the inputs differ. A normal `==` returns early at the first mismatch, which can leak information through timing.

In [ ]:
def verify(key: bytes, payload: bytes, received_tag: str) -> bool:
    expected = hmac.new(key, payload, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected, received_tag)

print(verify(key, payload, tag))                                   # True
print(verify(key, payload.replace(b"19.99", b"0.01"), tag))        # False — tampered

## Password hashing: never plain SHA-256

⚠️ **Gotcha:** `sha256(password)` is **not** password hashing. General-purpose hashes are designed to be *fast* — a GPU can try billions of guesses per second — and without a salt, identical passwords get identical hashes (rainbow tables).

Use a **key-derivation function** that is deliberately slow and salted:

- `hashlib.scrypt` — memory-hard; a strong choice (no BCL equivalent)
- `hashlib.pbkdf2_hmac` — the same algorithm as .NET's `Rfc2898DeriveBytes`; use a high iteration count (OWASP currently recommends 600,000 for PBKDF2-HMAC-SHA256)

```csharp
// C#
byte[] salt = RandomNumberGenerator.GetBytes(16);
byte[] hash = Rfc2898DeriveBytes.Pbkdf2(password, salt, 600_000, HashAlgorithmName.SHA256, 32);
```

In [ ]:
import secrets

password = "correct horse battery staple"     # an example only
salt = secrets.token_bytes(16)                 # unique random salt per password

pbkdf2_hash = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, 600_000)
scrypt_hash = hashlib.scrypt(password.encode(), salt=salt, n=2**14, r=8, p=1, dklen=32)

print(len(pbkdf2_hash), len(scrypt_hash))
# Store salt + parameters + hash together; the salt is not secret.

A minimal hash/verify pair — the salt and parameters are stored alongside the hash so you can verify later (and increase the cost over time):

In [ ]:
def hash_password(password: str) -> str:
    salt = secrets.token_bytes(16)
    digest = hashlib.scrypt(password.encode(), salt=salt, n=2**14, r=8, p=1, dklen=32)
    return f"scrypt$16384$8$1${salt.hex()}${digest.hex()}"

def check_password(password: str, stored: str) -> bool:
    _, n, r, p, salt_hex, digest_hex = stored.split("$")
    digest = hashlib.scrypt(password.encode(), salt=bytes.fromhex(salt_hex),
                            n=int(n), r=int(r), p=int(p), dklen=32)
    return hmac.compare_digest(digest.hex(), digest_hex)

stored = hash_password("s3cret-example")
print(stored.split("$")[:4])                    # algorithm and parameters
print(check_password("s3cret-example", stored)) # True
print(check_password("wrong-guess", stored))    # False

**Tip:** in a real application consider a dedicated library (e.g. `argon2-cffi` for Argon2id) — but if you are limited to the standard library, `scrypt` or a high-iteration `pbkdf2_hmac` are the right tools. For generating tokens and salts, use `secrets` *(see Standard Library › Numbers & Math › 006 · Secure Randomness with secrets)*.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Bytes → Base64 text | `Convert.ToBase64String(b)` | `base64.b64encode(b).decode("ascii")` |
| Base64 text → bytes | `Convert.FromBase64String(s)` | `base64.b64decode(s, validate=True)` |
| URL-safe Base64 | `Base64Url.EncodeToString(b)` | `base64.urlsafe_b64encode(b).rstrip(b"=")` |
| SHA-256 of bytes | `SHA256.HashData(b)` | `hashlib.sha256(b).digest()` |
| Hex string | `Convert.ToHexStringLower(h)` | `.hexdigest()` / `bytes.hex()` |
| Hash a stream | `SHA256.HashData(stream)` | `hashlib.file_digest(f, "sha256")` |
| Incremental hash | `IncrementalHash.AppendData` | `h.update(chunk)` |
| HMAC | `HMACSHA256.HashData(key, b)` | `hmac.digest(key, b, "sha256")` |
| Constant-time compare | `CryptographicOperations.FixedTimeEquals` | `hmac.compare_digest(a, b)` |
| Password hash | `Rfc2898DeriveBytes.Pbkdf2(...)` | `hashlib.scrypt(...)` / `hashlib.pbkdf2_hmac(...)` |

## Exercises

1. Write `to_b64url(data: bytes) -> str` and `from_b64url(text: str) -> bytes` that produce/accept **unpadded** URL-safe Base64, and check the round trip for byte strings of length 0 to 5.
2. Inside a `TemporaryDirectory`, write two files with identical content and one with different content. Use `hashlib.file_digest` to group file names by their SHA-256 (a mini duplicate finder).
3. Create a signed token `"<payload_b64url>.<hmac_hex>"` for the payload `b'{"user": "alice"}'`, and write a `read_token` function that returns the payload only if the signature is valid (otherwise raise `ValueError`).

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import base64, hashlib, hmac, tempfile
from collections import defaultdict
from pathlib import Path

# 1 — strip padding on the way out, restore it on the way in
def to_b64url(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode("ascii")

def from_b64url(text: str) -> bytes:
    return base64.urlsafe_b64decode(text + "=" * (-len(text) % 4))

print(all(from_b64url(to_b64url(bytes(range(n)))) == bytes(range(n)) for n in range(6)))  # True

# 2 — file_digest needs a binary file object
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "a.txt").write_bytes(b"same")
    (root / "b.txt").write_bytes(b"same")
    (root / "c.txt").write_bytes(b"different")
    groups = defaultdict(list)
    for path in sorted(root.iterdir()):
        with path.open("rb") as f:
            groups[hashlib.file_digest(f, "sha256").hexdigest()].append(path.name)
    print(sorted(groups.values()))   # [['a.txt', 'b.txt'], ['c.txt']]

# 3 — sign, then verify with compare_digest before trusting the payload
KEY = b"demo-key-not-a-real-secret"

def make_token(payload: bytes) -> str:
    return f"{to_b64url(payload)}.{hmac.new(KEY, payload, hashlib.sha256).hexdigest()}"

def read_token(token: str) -> bytes:
    body, sig = token.split(".")
    payload = from_b64url(body)
    expected = hmac.new(KEY, payload, hashlib.sha256).hexdigest()
    if not hmac.compare_digest(expected, sig):
        raise ValueError("bad signature")
    return payload

token = make_token(b'{"user": "alice"}')
print(read_token(token))
try:
    read_token(to_b64url(b'{"user": "admin"}') + "." + token.split(".")[1])
except ValueError as e:
    print(f"{type(e).__name__}: {e}")
```

</details>